# Credit Review Assistant — A100 멀티모달 vLLM 서버

Google Drive를 사용하지 않습니다. Python 패키지는 새 Colab 런타임마다 `/content/packages`에 설치하고, Qwen2.5-VL-32B-Instruct-AWQ 모델은 Hugging Face에서 `/content/models`로 직접 다운로드합니다.

같은 런타임에서는 완료 마커를 확인해 패키지와 모델을 재사용하며, 런타임이 초기화되면 다시 설치·다운로드합니다.

사전 준비:

1. Colab 런타임에서 GPU를 선택합니다.
2. Colab `보안 비밀`에 `NGROK_AUTH_TOKEN`, `COLAB_LLM_API_KEY`를 등록합니다.
3. 비공개 Hugging Face 모델을 사용할 경우 `HF_TOKEN`도 등록합니다.

`COLAB_LLM_API_KEY`는 Codespaces Secret에도 같은 값으로 등록하세요.

In [ ]:
# 1. Colab 로컬 런타임 기본 설정 (Google Drive 미사용)
import os
import sys
import json
import time
import socket
import shutil
import signal
import subprocess
import platform
from pathlib import Path

from google.colab import userdata

RUNTIME_ROOT = Path("/content/CreditReviewAssistant")
MODEL_ROOT = Path("/content/models")
LOG_ROOT = Path("/content/logs")

MODEL_ID = "Qwen/Qwen2.5-VL-32B-Instruct-AWQ"
MODEL_DIR = MODEL_ROOT / "Qwen2.5-VL-32B-Instruct-AWQ"
SERVED_MODEL_NAME = "credit-review-qwen-vl-32b"

HOST = "0.0.0.0"
PORT = 8000
MAX_MODEL_LEN = 8192
GPU_MEMORY_UTILIZATION = 0.90
# vLLM 자동 계산값(49.37 GiB)이 목표값(49.14 GiB)을 초과해
# EngineCore가 종료된 사례를 방지하기 위해 48 GiB로 명시합니다.
KV_CACHE_MEMORY_BYTES = 48 * 1024**3
MAX_NUM_SEQS = 2
MAX_IMAGES_PER_PROMPT = 2
FAST_STARTUP_MODE = True  # CUDA 그래프 생략: 시작은 빠르고 추론은 다소 느릴 수 있음

for directory in (RUNTIME_ROOT, MODEL_ROOT, LOG_ROOT):
    directory.mkdir(parents=True, exist_ok=True)

NGROK_AUTH_TOKEN = userdata.get("NGROK_AUTH_TOKEN")
COLAB_LLM_API_KEY = userdata.get("COLAB_LLM_API_KEY")
# 공개 모델은 HF_TOKEN 없이 다운로드할 수 있습니다.
# Secret이 등록되지 않은 경우에도 노트북 실행을 중단하지 않습니다.
try:
    HF_TOKEN = userdata.get("HF_TOKEN")
except userdata.SecretNotFoundError:
    HF_TOKEN = None

if not NGROK_AUTH_TOKEN:
    raise RuntimeError("Colab 보안 비밀에 NGROK_AUTH_TOKEN을 등록하세요.")
if not COLAB_LLM_API_KEY:
    raise RuntimeError("Colab 보안 비밀에 COLAB_LLM_API_KEY를 등록하세요.")

# 실행 중 생성되는 캐시도 Drive가 아닌 Colab 로컬 SSD에 둡니다.
os.environ["HF_HOME"] = "/content/hf_cache"
os.environ["TORCHINDUCTOR_CACHE_DIR"] = "/content/torchinductor_cache"
os.environ["VLLM_CACHE_ROOT"] = "/content/vllm_cache"
os.environ["HF_HUB_ENABLE_HF_TRANSFER"] = "1"
os.environ["TOKENIZERS_PARALLELISM"] = "false"

print(f"Colab 로컬 루트: {RUNTIME_ROOT}")
print(f"모델: {MODEL_ID}")

In [ ]:
# 2. GPU 및 런타임 확인
import torch

if not torch.cuda.is_available():
    raise RuntimeError("GPU 런타임이 아닙니다. 런타임 유형에서 GPU를 선택하세요.")

gpu_name = torch.cuda.get_device_name(0)
gpu_memory_gb = torch.cuda.get_device_properties(0).total_memory / (1024 ** 3)
cuda_version = torch.version.cuda or "unknown"

print(f"GPU: {gpu_name}")
print(f"VRAM: {gpu_memory_gb:.1f} GB")
print(f"PyTorch: {torch.__version__}, CUDA: {cuda_version}")

if gpu_memory_gb < 38:
    raise RuntimeError(
        "Qwen2.5-VL-32B-AWQ는 A100 40GB 이상의 GPU를 권장합니다. "
        f"현재 GPU={gpu_name}, VRAM={gpu_memory_gb:.1f}GB"
    )

if "A100" not in gpu_name and "H100" not in gpu_name and "RTX PRO 6000" not in gpu_name:
    print(
        f"주의: 현재 GPU는 {gpu_name}입니다. "
        "이 노트북은 A100 40GB 이상을 기준으로 설정되었습니다."
    )

runtime_key = (
    f"py{sys.version_info.major}{sys.version_info.minor}"
    f"-torch{torch.__version__.split('+')[0]}"
    f"-cu{cuda_version.replace('.', '')}"
)
PACKAGES_DIR = Path("/content/packages/credit-review-llm")
REQUIREMENTS_FILE = Path("/content/requirements-colab-llm.txt")
INSTALL_MARKER = PACKAGES_DIR / ".install_complete.json"

print(f"패키지 환경: {runtime_key}")

In [ ]:
# 3. 새 Colab 런타임의 로컬 SSD에 패키지 설치
# 같은 런타임에서는 완료 마커가 일치하면 재설치하지 않습니다.
import hashlib

REQUIREMENTS = (
    "vllm==0.27.1\n"
    "transformers>=4.49.0\n"
    "accelerate>=1.2.0\n"
    "qwen-vl-utils>=0.0.10\n"
    "huggingface-hub>=0.25.0\n"
    "hf-transfer>=0.1.8\n"
    "pyngrok>=7.2.0\n"
    "requests>=2.32.0\n"
    "pillow>=10.4.0\n"
)
REQUIREMENTS_FILE.write_text(REQUIREMENTS, encoding="utf-8")
requirements_hash = hashlib.sha256(REQUIREMENTS.encode("utf-8")).hexdigest()

installed_hash = None
if INSTALL_MARKER.exists():
    try:
        installed_hash = json.loads(
            INSTALL_MARKER.read_text(encoding="utf-8")
        ).get("requirements_sha256")
    except Exception:
        installed_hash = None

if installed_hash != requirements_hash:
    print("LLM 패키지를 Colab 로컬 SSD에 설치합니다. 새 런타임 최초 1회만 실행됩니다.")
    install_started_at = time.time()
    PACKAGES_DIR.mkdir(parents=True, exist_ok=True)
    subprocess.run(
        [
            sys.executable, "-m", "pip", "install",
            "--upgrade", "--target", str(PACKAGES_DIR),
            "--requirement", str(REQUIREMENTS_FILE),
        ],
        check=True,
    )
    INSTALL_MARKER.write_text(
        json.dumps(
            {
                "runtime_key": runtime_key,
                "requirements_sha256": requirements_hash,
                "installed_at": time.strftime("%Y-%m-%d %H:%M:%S"),
            },
            ensure_ascii=False,
            indent=2,
        ),
        encoding="utf-8",
    )
    print(f"로컬 패키지 설치 완료: {time.time() - install_started_at:.1f}초")
else:
    print("현재 Colab 런타임에 설치된 패키지를 재사용합니다.")

# 현재 노트북과 자식 vLLM 프로세스에서 로컬 패키지를 우선 사용합니다.
if str(PACKAGES_DIR) not in sys.path:
    sys.path.insert(0, str(PACKAGES_DIR))

existing_pythonpath = os.environ.get("PYTHONPATH", "")
os.environ["PYTHONPATH"] = (
    str(PACKAGES_DIR)
    if not existing_pythonpath
    else f"{PACKAGES_DIR}{os.pathsep}{existing_pythonpath}"
)

print(f"로컬 패키지 경로: {PACKAGES_DIR}")

In [ ]:
# 4. Hugging Face에서 모델을 Colab 로컬 SSD로 직접 다운로드
from huggingface_hub import snapshot_download

MODEL_REVISION = "66c370b74a18e7b1e871c97918f032ed3578dfef"
MODEL_MARKER = MODEL_DIR / ".download_complete.json"
required_model_files = ["config.json", "tokenizer_config.json", "preprocessor_config.json"]

def local_model_is_complete():
    if not MODEL_MARKER.is_file():
        return False
    if not all((MODEL_DIR / name).is_file() for name in required_model_files):
        return False
    return len(list(MODEL_DIR.glob("*.safetensors"))) == 6

if not local_model_is_complete():
    free_gb = shutil.disk_usage("/content").free / 1024**3
    if free_gb < 30:
        raise RuntimeError(f"모델 다운로드 공간이 부족합니다: {free_gb:.1f}GB")
    print("Hugging Face에서 모델을 Colab 로컬 SSD로 다운로드합니다.")
    download_started_at = time.time()
    MODEL_DIR.mkdir(parents=True, exist_ok=True)
    snapshot_download(
        repo_id=MODEL_ID,
        revision=MODEL_REVISION,
        local_dir=str(MODEL_DIR),
        token=HF_TOKEN or None,
    )
    if not all((MODEL_DIR / name).is_file() for name in required_model_files):
        raise RuntimeError("모델 설정 파일 다운로드가 불완전합니다.")
    safetensors = list(MODEL_DIR.glob("*.safetensors"))
    if len(safetensors) != 6:
        raise RuntimeError(f"모델 가중치 다운로드가 불완전합니다: {len(safetensors)}/6")
    MODEL_MARKER.write_text(
        json.dumps(
            {
                "model_id": MODEL_ID,
                "revision": MODEL_REVISION,
                "downloaded_at": time.strftime("%Y-%m-%d %H:%M:%S"),
            },
            ensure_ascii=False,
            indent=2,
        ),
        encoding="utf-8",
    )
    print(f"모델 다운로드 완료: {time.time() - download_started_at:.1f}초")
else:
    print("현재 Colab 런타임의 다운로드된 모델을 재사용합니다.")

print(f"실행 모델 경로: {MODEL_DIR}")
print(f"safetensors 조각 수: {len(list(MODEL_DIR.glob('*.safetensors')))}")

In [ ]:
# 5. 기존 서버와 ngrok 터널 정리
import requests
from pyngrok import ngrok, conf

server_process = globals().get("server_process")
if server_process is not None and server_process.poll() is None:
    print("기존 vLLM 서버를 종료합니다.")
    server_process.terminate()
    try:
        server_process.wait(timeout=20)
    except subprocess.TimeoutExpired:
        server_process.kill()

try:
    ngrok.kill()
except Exception:
    pass

# 이전 실행에서 포트가 남았는지 확인합니다.
with socket.socket(socket.AF_INET, socket.SOCK_STREAM) as sock:
    if sock.connect_ex(("127.0.0.1", PORT)) == 0:
        raise RuntimeError(
            f"{PORT}번 포트가 이미 사용 중입니다. 런타임을 재시작한 뒤 다시 실행하세요."
        )

In [ ]:
# 6. vLLM OpenAI 호환 서버 실행
SERVER_LOG = LOG_ROOT / "vllm-server.log"
log_handle = open(SERVER_LOG, "w", encoding="utf-8")

server_command = [
    sys.executable,
    "-m",
    "vllm.entrypoints.openai.api_server",
    "--host",
    HOST,
    "--port",
    str(PORT),
    "--model",
    str(MODEL_DIR),
    "--served-model-name",
    SERVED_MODEL_NAME,
    "--api-key",
    COLAB_LLM_API_KEY,
    "--quantization",
    "awq",
    "--dtype",
    "auto",
    "--max-model-len",
    str(MAX_MODEL_LEN),
    "--gpu-memory-utilization",
    str(GPU_MEMORY_UTILIZATION),
    "--kv-cache-memory",
    str(KV_CACHE_MEMORY_BYTES),
    "--max-num-seqs",
    str(MAX_NUM_SEQS),
    "--limit-mm-per-prompt",
    json.dumps(
        {
            "image": MAX_IMAGES_PER_PROMPT,
            "video": 0,
        }
    ),
    "--trust-remote-code",
]

if FAST_STARTUP_MODE:
    server_command.append("--enforce-eager")

server_env = os.environ.copy()
server_process = subprocess.Popen(
    server_command,
    stdout=log_handle,
    stderr=subprocess.STDOUT,
    env=server_env,
)

print(f"vLLM 서버 시작 중... PID={server_process.pid}")
print(f"로그: {SERVER_LOG}")

health_url = f"http://127.0.0.1:{PORT}/health"
startup_timeout_seconds = 3600
started_at = time.time()

while True:
    if server_process.poll() is not None:
        log_handle.flush()
        tail = SERVER_LOG.read_text(encoding="utf-8", errors="replace")[-30000:]
        raise RuntimeError(f"vLLM 서버가 종료되었습니다. 최근 로그:\n{tail}")

    try:
        response = requests.get(health_url, timeout=5)
        if response.ok:
            break
    except requests.RequestException:
        pass

    elapsed = int(time.time() - started_at)
    if elapsed > startup_timeout_seconds:
        server_process.terminate()
        raise TimeoutError(f"vLLM 서버가 {startup_timeout_seconds}초 안에 준비되지 않았습니다.")

    if elapsed % 30 < 5:
        print(f"모델 로딩 대기 중... {elapsed}초")
    time.sleep(5)

print("vLLM OpenAI 호환 서버가 준비되었습니다.")

In [ ]:
# 7. ngrok HTTPS 터널 생성
ngrok.set_auth_token(NGROK_AUTH_TOKEN)
tunnel = ngrok.connect(addr=PORT, proto="http")
public_url = tunnel.public_url.rstrip("/")

print("=" * 70)
print("Codespaces 설정값")
print(f"COLAB_LLM_BASE_URL={public_url}/v1")
print(f"COLAB_LLM_MODEL={SERVED_MODEL_NAME}")
print("COLAB_LLM_API_KEY=<Colab Secret과 동일한 값>")
print("=" * 70)

In [ ]:
# 8. 텍스트 API 테스트
test_response = requests.post(
    f"http://127.0.0.1:{PORT}/v1/chat/completions",
    headers={
        "Authorization": f"Bearer {COLAB_LLM_API_KEY}",
        "Content-Type": "application/json",
    },
    json={
        "model": SERVED_MODEL_NAME,
        "messages": [
            {
                "role": "system",
                "content": "당신은 금융기관의 신중한 여신심사 보조자입니다.",
            },
            {
                "role": "user",
                "content": "부채비율이 높을 때 확인해야 할 핵심 항목을 세 가지로 답하세요.",
            },
        ],
        "temperature": 0.1,
        "top_p": 0.8,
        "max_tokens": 256,
        "extra_body": {"repetition_penalty": 1.05},
    },
    timeout=180,
)
test_response.raise_for_status()
print(test_response.json()["choices"][0]["message"]["content"])

In [ ]:
# 9. 공개 이미지 URL로 멀티모달 API 테스트
test_image_url = (
    "https://huggingface.co/datasets/huggingface/"
    "documentation-images/resolve/main/p-blog/candy.JPG"
)

vision_response = requests.post(
    f"http://127.0.0.1:{PORT}/v1/chat/completions",
    headers={
        "Authorization": f"Bearer {COLAB_LLM_API_KEY}",
        "Content-Type": "application/json",
    },
    json={
        "model": SERVED_MODEL_NAME,
        "messages": [
            {
                "role": "system",
                "content": (
                    "당신은 금융 문서와 이미지를 분석하는 신중한 "
                    "여신심사 보조자입니다. 확실하지 않은 내용은 추측하지 마세요."
                ),
            },
            {
                "role": "user",
                "content": [
                    {
                        "type": "text",
                        "text": "이미지에 보이는 내용을 한국어로 설명하세요.",
                    },
                    {
                        "type": "image_url",
                        "image_url": {"url": test_image_url},
                    },
                ],
            },
        ],
        "temperature": 0.1,
        "top_p": 0.8,
        "max_tokens": 512,
        "extra_body": {"repetition_penalty": 1.05},
    },
    timeout=300,
)
vision_response.raise_for_status()
print(vision_response.json()["choices"][0]["message"]["content"])

In [ ]:
# 10. Colab 로컬 이미지 파일을 Base64 data URL로 변환하는 도우미
import base64
import mimetypes

def image_to_data_url(image_path: str) -> str:
    path = Path(image_path)
    if not path.is_file():
        raise FileNotFoundError(path)

    mime_type, _ = mimetypes.guess_type(path.name)
    mime_type = mime_type or "image/png"
    encoded = base64.b64encode(path.read_bytes()).decode("utf-8")
    return f"data:{mime_type};base64,{encoded}"

# 사용 예:
# local_image_url = image_to_data_url("/content/sample_screenshot.png")
# 위 URL을 9번 셀의 image_url.url 값에 넣어 요청하면 됩니다.
print("로컬 이미지 변환 도우미가 준비되었습니다.")

## Codespaces 연결 예시

```python
from langchain_openai import ChatOpenAI

llm = ChatOpenAI(
    base_url=settings.COLAB_LLM_BASE_URL,
    api_key=settings.COLAB_LLM_API_KEY,
    model=settings.COLAB_LLM_MODEL,
    temperature=0.2,
    timeout=180,
    max_retries=1,
)
```

Colab 세션이 종료되면 터널 URL도 바뀔 수 있습니다. 새 실행 시 출력된 `COLAB_LLM_BASE_URL`을 Codespaces Secret 또는 환경 변수에 갱신하세요.